# 05 — Agent Tools
Creates the governed Unity Catalog tools and the ML-backed next-day demand tool used by the LangGraph agent.

**Unity Catalog tools**
- `pgcb.tools.get_grid_data`
- `pgcb.tools.calculate_supply_gap`
- `pgcb.tools.recommend_grid_action`

**Local ML tool**
- `forecast_next_day_demand`

> Prerequisites: `unitycatalog-ai[databricks]`, `databricks-langchain`, `langgraph`, and `mlflow` must be installed on the compute.

In [ ]:
import pandas as pd
import mlflow
import mlflow.pyfunc

from unitycatalog.ai.core.databricks import DatabricksFunctionClient
from databricks_langchain import UCFunctionToolkit
from langchain_core.tools import tool

TOOLS_SCHEMA = "pgcb.tools"
FORECAST_MODEL_URI = "models:/pgcb.models.holt_winters_demand@champion"

spark.sql("CREATE SCHEMA IF NOT EXISTS pgcb.tools")
mlflow.set_registry_uri("databricks-uc")

## 1. Register deterministic calculation tool

In [ ]:
client = DatabricksFunctionClient()

def calculate_supply_gap(
    predicted_demand_mw: float,
    expected_generation_mw: float,
    expected_imports_mw: float
) -> str:
    """Calculate supply gap and reserve margin for a forecast scenario.

    Args:
        predicted_demand_mw: Forecast electricity demand in MW.
        expected_generation_mw: Expected domestic generation in MW.
        expected_imports_mw: Expected imported electricity in MW.

    Returns:
        Semicolon-delimited calculation results including supply gap and reserve margin.
    """
    if predicted_demand_mw <= 0:
        return "ERROR: Predicted demand must be greater than zero."
    if expected_generation_mw < 0 or expected_imports_mw < 0:
        return "ERROR: Generation and imports cannot be negative."

    total_supply = expected_generation_mw + expected_imports_mw
    gap = total_supply - predicted_demand_mw
    reserve_margin = (gap / predicted_demand_mw) * 100

    return (
        f"Predicted demand: {predicted_demand_mw:.2f} MW; "
        f"Expected generation: {expected_generation_mw:.2f} MW; "
        f"Expected imports: {expected_imports_mw:.2f} MW; "
        f"Total supply: {total_supply:.2f} MW; "
        f"Supply gap: {gap:.2f} MW; "
        f"Reserve margin: {reserve_margin:.2f}%"
    )

client.create_python_function(
    func=calculate_supply_gap,
    catalog="pgcb",
    schema="tools",
    replace=True,
)

## 2. Register rule-based recommendation tool

In [ ]:
def recommend_grid_action(
    supply_gap_mw: float,
    reserve_margin_pct: float
) -> str:
    """Classify projected grid risk and return a project-defined recommendation.

    Args:
        supply_gap_mw: Expected supply minus forecast demand in MW.
        reserve_margin_pct: Expected reserve margin percentage.

    Returns:
        Risk level and recommended operational response.
    """
    if reserve_margin_pct >= 10:
        return (
            "Risk level: LOW. Expected supply is comfortably above forecast demand. "
            "Continue normal monitoring."
        )
    if reserve_margin_pct >= 5:
        return (
            "Risk level: MODERATE. Supply is adequate but reserve margin is reduced. "
            "Monitor demand and generation conditions closely."
        )
    if reserve_margin_pct >= 0:
        return (
            "Risk level: HIGH. Supply exceeds demand only by a small margin. "
            "Prepare additional generation or import capacity as contingency."
        )
    return (
        "Risk level: CRITICAL. Expected supply is below forecast demand. "
        "Evaluate additional generation, imports, or other approved balancing measures."
    )

client.create_python_function(
    func=recommend_grid_action,
    catalog="pgcb",
    schema="tools",
    replace=True,
)

## 3. Create historical grid-summary tool

In [ ]:
spark.sql("""
CREATE OR REPLACE FUNCTION pgcb.tools.get_grid_data(
    start_date DATE,
    end_date DATE
)
RETURNS TABLE (
    avg_demand_mw DOUBLE,
    peak_demand_mw DOUBLE,
    avg_generation_mw DOUBLE,
    avg_imports_mw DOUBLE,
    avg_load_shedding_mw DOUBLE
)
RETURN
SELECT
    AVG(demand_mw) AS avg_demand_mw,
    MAX(demand_mw) AS peak_demand_mw,
    AVG(generation_mw) AS avg_generation_mw,
    AVG(total_imports_mw) AS avg_imports_mw,
    AVG(load_shedding) AS avg_load_shedding_mw
FROM pgcb.gold.grid_operations
WHERE date BETWEEN start_date AND end_date
""")

## 4. Load the registered forecasting model and expose it as a LangChain tool

In [ ]:
forecast_model = mlflow.pyfunc.load_model(FORECAST_MODEL_URI)

@tool
def forecast_next_day_demand() -> float:
    """Forecast average electricity demand for the next day.

    Returns:
        Next-day average electricity demand in MW as a numeric value.
    """
    prediction = forecast_model.predict(
        pd.DataFrame({"days_ahead": [1]})
    )
    return float(prediction.iloc[0]["predicted_demand_mw"])

## 5. Build the agent tool list

In [ ]:
toolkit = UCFunctionToolkit(
    function_names=[
        "pgcb.tools.get_grid_data",
        "pgcb.tools.calculate_supply_gap",
        "pgcb.tools.recommend_grid_action",
    ]
)

tools = toolkit.tools + [forecast_next_day_demand]

print(f"Number of tools: {len(tools)}")
for agent_tool in tools:
    print(agent_tool.name)

## 6. Smoke-test the forecasting tool

In [ ]:
print(f"Next-day forecast: {forecast_next_day_demand.invoke({}):.2f} MW")